In [3]:
import pandas as pd
import graphistry
import os
from pathlib import Path

In [4]:
GRAPHISTRY_KEY_ID = "91LX0MJCAF"   #os.environ["GRAPHISTRY_KEY_ID"]
GRAPHISTRY_API_KEY = "D76N4294TL52ELOB" #os.environ["GRAPHISTRY_API_KEY"]

In [5]:
graphistry.register(api=3, server='hub.graphistry.com', personal_key_id=GRAPHISTRY_KEY_ID, personal_key_secret=GRAPHISTRY_API_KEY)

In [6]:

def load_knowledge_graph(data_folder='./'):
    """
    Load nodes and edges from CSV files to create a knowledge graph
    
    Args:
        data_folder: Path to folder containing the CSV files
    """
    
    # Load nodes (companies)
    companies_path = Path(data_folder) / 'companies.csv'
    nodes_df = pd.read_csv(companies_path)
    
    print(f"Loaded {len(nodes_df)} companies")
    print("Sample companies:")
    print(nodes_df.head())
    
    # Define edge file names and their relationship types
    edge_files = [
        'has_supplier.csv',
        'has_investor.csv', 
        'has_subsidiary.csv',
        'subsidiary_of.csv',
        'invests_in.csv',
        'supplies.csv',
        'partnered_with.csv'
    ]
    
    # Load and combine all edge files
    all_edges = []
    
    for edge_file in edge_files:
        edge_path = Path(data_folder) / edge_file
        
        if edge_path.exists():
            # Extract relationship type from filename
            relationship_type = edge_file.replace('.csv', '')
            
            # Load edges
            edges_df = pd.read_csv(edge_path)
            
            # Rename columns to standard format
            edges_df.columns = ['source', 'target']
            
            # Add relationship type
            edges_df['relationship'] = relationship_type
            
            all_edges.append(edges_df)
            print(f"Loaded {len(edges_df)} {relationship_type} relationships")
        else:
            print(f"Warning: {edge_file} not found")
    
    # Combine all edges
    if all_edges:
        combined_edges_df = pd.concat(all_edges, ignore_index=True)
        print(f"\nTotal edges: {len(combined_edges_df)}")
        print("Relationship type distribution:")
        print(combined_edges_df['relationship'].value_counts())
    else:
        print("No edge files found!")
        return None, None
    
    return nodes_df, combined_edges_df


def analyze_graph_structure(nodes_df, edges_df):
    """
    Print basic statistics about the graph structure
    """
    print("\n" + "="*50)
    print("GRAPH ANALYSIS")
    print("="*50)
    
    print(f"Number of nodes (companies): {len(nodes_df)}")
    print(f"Number of edges (relationships): {len(edges_df)}")
    
    # Public vs Private companies
    if 'ticker' in nodes_df.columns:
        public_companies = nodes_df['ticker'].notna().sum()
        private_companies = len(nodes_df) - public_companies
        print(f"Public companies (with ticker): {public_companies}")
        print(f"Private companies: {private_companies}")
    
    # Most connected companies
    all_companies = pd.concat([edges_df['source'], edges_df['target']])
    company_connections = all_companies.value_counts().head(10)
    
    print(f"\nTop 10 most connected companies:")
    for company, count in company_connections.items():
        company_name = nodes_df[nodes_df['id'] == company]['name'].iloc[0] if len(nodes_df[nodes_df['id'] == company]) > 0 else company
        print(f"  {company_name}: {count} connections")
    
    # Relationship types
    print(f"\nRelationship types:")
    for rel_type, count in edges_df['relationship'].value_counts().items():
        print(f"  {rel_type}: {count}")


In [8]:
def create_graphistry_visualization(nodes_df, edges_df):
    """
    Create and display Graphistry visualization
    """
    
    # Create a copy to avoid modifying original dataframes
    nodes_working = nodes_df.copy()
    edges_working = edges_df.copy()
    
    # Style nodes based on whether they have ticker symbols
    # Companies with tickers are likely public companies
    if 'ticker' in nodes_working.columns:
        def node_color(row):
            if pd.notna(row['ticker']):
                return '#00FF00'  # Green for public companies
            else:
                return '#0066CC'  # Blue for private companies
        
        nodes_working['node_color'] = nodes_working.apply(node_color, axis=1)
    
    # Style edges by relationship type
    relationship_colors = {
        'has_supplier': '#FF6B6B',      # Red
        'has_investor': '#4ECDC4',      # Teal  
        'has_subsidiary': '#45B7D1',    # Blue
        'subsidiary_of': '#96CEB4',     # Green
        'invests_in': '#FECA57',        # Yellow
        'supplies': '#FF9FF3',          # Pink
        'partnered_with': '#A55EEA'     # Purple
    }
    
    def get_edge_color(relationship):
        return relationship_colors.get(relationship, '#999999')  # Default gray
    
    edges_working['edge_color'] = edges_working['relationship'].apply(get_edge_color)
    
    # Calculate node degrees (number of connections)
    degree_counts = pd.concat([
        edges_working['source'].value_counts(),
        edges_working['target'].value_counts()
    ], axis=1, sort=False).fillna(0).sum(axis=1)
    
    # Merge degree counts back to nodes
    nodes_working = nodes_working.merge(
        degree_counts.reset_index().rename(columns={'index': 'id', 0: 'degree'}),
        on='id', 
        how='left'
    ).fillna({'degree': 0})
    
    # Create the graph object using bind method
    g = graphistry.bind(source='source', destination='target', node='id')
    g = g.edges(edges_working)
    g = g.nodes(nodes_working)
    
    # Try different encoding methods based on Graphistry version
    try:
        # Newer API style
        if 'node_color' in nodes_working.columns:
            g = g.encode_point_color('node_color')
        if 'edge_color' in edges_working.columns:
            g = g.encode_edge_color('edge_color')
        if 'name' in nodes_working.columns:
            g = g.encode_point_label('name')
        if 'relationship' in edges_working.columns:
            g = g.encode_edge_label('relationship')
        if 'degree' in nodes_working.columns:
            g = g.encode_point_size('degree')
            
    except AttributeError:
        try:
            # Alternative API style
            if 'node_color' in nodes_working.columns:
                g = g.encode_node_color('node_color')
            if 'edge_color' in edges_working.columns:
                g = g.encode_edge_color('edge_color')
            if 'name' in nodes_working.columns:
                g = g.encode_node_label('name')
            if 'relationship' in edges_working.columns:
                g = g.encode_edge_label('relationship')
            if 'degree' in nodes_working.columns:
                g = g.encode_node_size('degree')
                
        except AttributeError:
            # Fallback - basic graph without advanced encoding
            print("Note: Using basic visualization without advanced styling")
            pass
    
    # Configure layout settings if available
    try:
        g = g.settings(url_params={
            'play': 2000,  # Auto-layout for 2 seconds
            'strongGravity': True,
            'edgeInfluence': 1.0,
            'precisionVsSpeed': 1.0,
            'gravity': 0.1,
            'scalingRatio': 10.0
        })
    except:
        print("Note: Advanced layout settings not available in this Graphistry version")
    
    return g

In [9]:

nodes_df, edges_df = load_knowledge_graph('../data/sec/kuzu')  # Assumes CSV files are in current directory
    
if nodes_df is not None and edges_df is not None:
    # Analyze the graph structure
    analyze_graph_structure(nodes_df, edges_df)
        
    # Create and display the visualization
    print("\nCreating Graphistry visualization...")
    g = create_graphistry_visualization(nodes_df, edges_df)
        
        # Generate the visualization URL/display
    url = g.plot(render=False)  # Set render=True to open in browser automatically
    print(f"\nVisualization URL: {url}")

    # Optionally save the graph data for later use
    #g.pandas_edges().to_csv('processed_edges.csv', index=False)
    #g.pandas_nodes().to_csv('processed_nodes.csv', index=False)
    #print("\nProcessed data saved to processed_edges.csv and processed_nodes.csv")
        
else:
    print("Failed to load knowledge graph data.")

# Additional helper functions for specific analyses

def find_company_network(company_id, edges_df, nodes_df, max_depth=2):
    """
    Find all companies connected to a specific company within max_depth steps
    """
    visited = set()
    current_level = {company_id}
    
    for depth in range(max_depth):
        next_level = set()
        for company in current_level:
            if company not in visited:
                visited.add(company)
                # Find connected companies
                connected = set(edges_df[edges_df['source'] == company]['target'].tolist() + 
                              edges_df[edges_df['target'] == company]['source'].tolist())
                next_level.update(connected - visited)
        current_level = next_level
        if not current_level:
            break
    
    # Get company details
    network_companies = nodes_df[nodes_df['id'].isin(visited)]
    network_edges = edges_df[
        (edges_df['source'].isin(visited)) & 
        (edges_df['target'].isin(visited))
    ]
    
    return network_companies, network_edges

def create_subgraph_visualization(company_id, edges_df, nodes_df, max_depth=2):
    """
    Create a focused visualization around a specific company
    """
    sub_nodes, sub_edges = find_company_network(company_id, edges_df, nodes_df, max_depth)
    
    if len(sub_nodes) > 1:
        g = create_graphistry_visualization(sub_nodes, sub_edges)
        return g
    else:
        print(f"No network found for company: {company_id}")
        return None

Loaded 7067 companies
Sample companies:
                             id                                     name  \
0                           MIR                Mirion Technologies, Inc.   
1                    TRowePrice                            T. Rowe Price   
2          ec2SoftwareSolutions               ec2 Software Solutions LLC   
3          MirionIntermediateCo              Mirion IntermediateCo, Inc.   
4  MirionTechnologiesUSHoldings  Mirion Technologies (US Holdings), Inc.   

  ticker  
0    MIR  
1    NaN  
2    NaN  
3    NaN  
4    NaN  
Loaded 1929 has_supplier relationships
Loaded 1538 has_investor relationships
Loaded 2748 has_subsidiary relationships
Loaded 2748 subsidiary_of relationships
Loaded 1538 invests_in relationships
Loaded 1929 supplies relationships
Loaded 3105 partnered_with relationships

Total edges: 15535
Relationship type distribution:
relationship
partnered_with    3105
has_subsidiary    2748
subsidiary_of     2748
has_supplier      1929
supplies